# Modeling

In [63]:
import pandas as pd
from sklearn.decomposition import PCA
import numpy as np

from sklearn.linear_model import Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor,
    ExtraTreesRegressor
)

import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn import model_selection
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV


from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.metrics import root_mean_squared_log_error
from numpy import expm1
from pathlib import Path

In [64]:
# PATH DEFINITIONS
BASE_DIR = Path().resolve().parent

DATA_DIR = BASE_DIR / "data"
MODEL_DATA_DIR = DATA_DIR / "data_model"

In [65]:
df_train = pd.read_csv(MODEL_DATA_DIR / "train_model.csv")

In [66]:
X_train = df_train.drop(columns=['SALEPRICE_LOG'])
y_train = df_train['SALEPRICE_LOG']

In [67]:
df_train.head()

,ID,MSSUBCLASS,MSZONING,HASREGULARLOTSHAPE,NEIGHBORHOOD,HOUSESTYLE,OVERALLQUAL,YEARREMODADD,EXTERIOR1ST,EXTERQUAL,...,HASWOODDECKSF,WOODDECKSF_LOG,HASSCREENPORCH,HASLOWQUALFINSF,HASENCLOSEDPORCH,HASMASVNRAREA,MASVNRAREA_LOG,HAS3SSNPORCH,HASLOTFRONTAGE,SALEPRICE_LOG
0,1,12.340706,12.1001,True,12.164080,12.212548,2,5,12.219079,3,...,False,0.000000,False,False,False,True,5.283204,False,True,12.247699
1,2,12.067974,12.1001,True,12.074029,12.011076,2,31,11.878755,2,...,True,5.700444,False,False,False,False,0.000000,False,True,12.109016
2,3,12.340706,12.1001,False,12.164080,12.212548,2,6,12.219079,3,...,False,0.000000,False,False,False,True,5.093750,False,True,12.317171
3,4,11.972299,12.1001,False,12.224805,12.212548,2,36,11.844305,2,...,False,0.000000,False,False,True,False,0.000000,False,True,11.849405
4,5,12.340706,12.1001,False,12.659042,12.212548,3,8,12.219079,3,...,True,5.262690,False,False,False,True,5.860786,False,True,12.429220


In [68]:
models = {
    'Ridge':             Ridge(random_state=0),
    'Lasso':             Lasso(random_state=0),
    'ElasticNet':        ElasticNet(random_state=0),
    'Bayesian Ridge':    BayesianRidge(),
    'KNN Regressor':     KNeighborsRegressor(),
    'SVR':               SVR(),
    'Random Forest':     RandomForestRegressor(random_state=0),
    'Extra Trees':       ExtraTreesRegressor(random_state=0),
    'Gradient Boosting': GradientBoostingRegressor(random_state=0),
    'XGBoost:': xgb.XGBRegressor(random_state=0)
}

kfold = model_selection.KFold(n_splits=10, shuffle=True, random_state=0)

In [69]:
results = []
for name, model in models.items():
    pipeline = Pipeline([
        ('model', model)
    ])
    scores = model_selection.cross_validate(
        pipeline,
        X_train,
        y_train,
        scoring='neg_root_mean_squared_log_error',
        cv=kfold,
        n_jobs=-1
    )
    results.append({
        'MODEL': name.upper(),
        'RMSLE_MEAN': -scores['test_score'].mean(),
        'RMSLE_STD':   scores['test_score'].std()
    })

In [70]:
df_exploration = pd.DataFrame(results)
df_exploration.columns = df_exploration.columns.str.upper()
df_exploration = df_exploration.round(4)

df_exploration = df_exploration.sort_values(by='RMSLE_MEAN').reset_index(drop=True)

df_exploration

,MODEL,RMSLE_MEAN,RMSLE_STD
0,RIDGE,0.0110,0.0007
1,BAYESIAN RIDGE,0.0110,0.0007
2,GRADIENT BOOSTING,0.0110,0.0008
3,RANDOM FOREST,0.0115,0.0011
4,XGBOOST:,0.0115,0.0012
5,EXTRA TREES,0.0119,0.0012
6,ELASTICNET,0.0245,0.0015
7,SVR,0.0246,0.0016
8,LASSO,0.0247,0.0015
9,KNN REGRESSOR,0.0251,0.0016


## GRADIENT BOOSTING

In [71]:
df_exploration_analysis = df_exploration.copy()
df_exploration_analysis[['RMSLE_MEAN', 'RMSLE_STD']] = df_exploration_analysis[['RMSLE_MEAN', 'RMSLE_STD']].rank(0, numeric_only=True, method='min', ascending=True)
df_exploration_analysis['POINTS'] = df_exploration_analysis['RMSLE_MEAN'] + df_exploration_analysis['RMSLE_STD'] 

df_exploration_analysis = df_exploration_analysis.sort_values(by='POINTS', ascending=True).reset_index(drop=True)
df_exploration_analysis

,MODEL,RMSLE_MEAN,RMSLE_STD,POINTS
0,RIDGE,1.0,1.0,2.0
1,BAYESIAN RIDGE,1.0,1.0,2.0
2,GRADIENT BOOSTING,1.0,3.0,4.0
3,RANDOM FOREST,4.0,4.0,8.0
4,XGBOOST:,4.0,5.0,9.0
5,EXTRA TREES,6.0,5.0,11.0
6,ELASTICNET,7.0,7.0,14.0
7,LASSO,9.0,7.0,16.0
8,SVR,8.0,9.0,17.0
9,KNN REGRESSOR,10.0,9.0,19.0


## Bayesian Model

In [72]:
# bayesian_model = BayesianRidge()

# param_distributions_bayesian = {
#     "alpha_1": [1e-6, 1e-5, 1e-4],   # hiperparâmetro do prior gamma (precisão de alpha)
#     "alpha_2": [1e-6, 1e-5, 1e-4],
#     "lambda_1": [1e-6, 1e-5, 1e-4],  # hiperparâmetro do prior gamma (precisão de lambda/pesos)
#     "lambda_2": [1e-6, 1e-5, 1e-4],
#     "max_iter": [300, 500, 700],
#     "tol": [1e-4, 1e-3, 1e-5]
# }

# random_search_bayesian = RandomizedSearchCV(
#     estimator=bayesian_model,
#     param_distributions=param_distributions_bayesian,
#     n_jobs=-1,
#     random_state=0,
#     scoring='neg_root_mean_squared_log_error'
# )

# random_search_bayesian.fit(X_train, y_train)

# model_best_bayesian = random_search_bayesian.best_estimator_
# model_best_bayesian_score = random_search_bayesian.best_score_

# model_best_bayesian

In [73]:
# param_distributions_bayesian_refined = {
#     "alpha_1": [1e-4, 1e-3],
#     "alpha_2": [1e-5],
#     "lambda_1": [1e-6, 1e-7],
#     "lambda_2": [1e-3, 1e-4],
#     "max_iter": [200, 300, 400],
#     "tol": [1e-5]
# }

# grid_search = GridSearchCV(
#     estimator=bayesian_model,
#     param_grid=param_distributions_bayesian_refined,
#     n_jobs=-1,
#     scoring='neg_root_mean_squared_log_error',
#     return_train_score=True
# )

# grid_search.fit(X_train, y_train)

# model_best_bm = grid_search.best_estimator_
# model_best_bm_score = grid_search.best_score_
# y_pred_bm = model_best_bm.predict(X_train)


## Ridge Model

In [74]:
# ridge_model = Ridge(random_state=0)

# param_distributions_ridge = {
#     "alpha": [0.1, 0.5, 1.0, 5.0, 10.0, 20.0, 50.0],
#     "solver": ["auto", "svd", "cholesky", "lsqr"],
#     "fit_intercept": [True, False],
#     "tol": [1e-4, 1e-3, 1e-5]
# }

# random_search_ridge = RandomizedSearchCV(
#     estimator=ridge_model,
#     param_distributions=param_distributions_ridge,
#     n_jobs=-1,
#     random_state=0,
#     scoring='neg_root_mean_squared_log_error'
# )

# random_search_ridge.fit(X_train, y_train)

# model_best_ridge = random_search_ridge.best_estimator_
# model_best_ridge_score = random_search_ridge.best_score_

# print(model_best_ridge)

In [75]:
ridge_model = Ridge(random_state=0)

param_distributions_ridge_refined = {
    "alpha": [1.0, 2.0, 2.5, 3.0, 3.5],
    "solver": ["svd"],
    "fit_intercept": [True],
    "tol": [1e-4, 1e-3]
}

grid_search = GridSearchCV(
    estimator=ridge_model,
    param_grid=param_distributions_ridge_refined,
    n_jobs=-1,
    scoring='neg_root_mean_squared_log_error',
    return_train_score=True
)

grid_search.fit(X_train, y_train)

model_best_rm = grid_search.best_estimator_
model_best_rm_score = grid_search.best_score_
y_pred_rm = model_best_rm.predict(X_train)


print(model_best_rm)

Ridge(alpha=2.5, random_state=0, solver='svd')


## Gradient Boosting Regressor

In [76]:
# gb_model = GradientBoostingRegressor(random_state=0)

# param_distributions = {
#         "n_estimators": [500, 525, 550],
#         "learning_rate": [0.045, 0.05, 0.06],
#         "max_depth": [6, 7], # experimentacao
#         "min_samples_split": [90, 100, 110], #dobro do min_sample_leaf
#         "min_samples_leaf": [45, 50, 55], # Geralmente, na faixa de 1-5%
#         "max_features": [0.8, 0.85, 0.90], # um valor classico é \sqrt (no caso, 4 = 0.25)
#         "subsample": [0.80, 0.85, 0.90] # depende do tamanho do modelo, cria generalização
#     }

# random_search = RandomizedSearchCV(
#         estimator=gb_model,
#         param_distributions=param_distributions,
#         n_jobs=-1,
#         random_state=0,
#         scoring='neg_root_mean_squared_log_error'
#     )

# random_search.fit(X_train, y_train)

# model_best_gb = random_search.best_estimator_
# model_best_gb_score = random_search.best_score_

# model_best_gb

In [77]:
gb_model = GradientBoostingRegressor(random_state=0)

param_distributions_gb_refined = {
        "n_estimators": [550, 575, 600],
        "learning_rate": [0.045, 0.04],
        "max_depth": [6, 7], # experimentacao
        "min_samples_split": [90, 80, 70], #dobro do min_sample_leaf
        "min_samples_leaf": [45, 40, 45], # Geralmente, na faixa de 1-5%
        "max_features": [0.7, 0.75, 0.8 ], # um valor classico é \sqrt (no caso, 4 = 0.25)
        "subsample": [0.85] # depende do tamanho do modelo, cria generalização
    }


grid_search = GridSearchCV(
    estimator=gb_model,
    param_grid=param_distributions_gb_refined,
    n_jobs=-1,
    scoring='neg_root_mean_squared_log_error',
    return_train_score=True
)

grid_search.fit(X_train, y_train)

model_best_gb = grid_search.best_estimator_
model_best_gb_score = grid_search.best_score_
y_pred_gb = model_best_gb.predict(X_train)

print(model_best_gb)


GradientBoostingRegressor(learning_rate=0.045, max_depth=6, max_features=0.7,
                          min_samples_leaf=40, min_samples_split=90,
                          n_estimators=575, random_state=0, subsample=0.85)


In [78]:
# rmsle_bm = root_mean_squared_log_error(y_true=y_train, y_pred=y_pred_bm)
rmsle_rm = root_mean_squared_log_error(y_true=y_train, y_pred=y_pred_rm)
rmsle_gb = root_mean_squared_log_error(y_true=y_train, y_pred=y_pred_gb)

# w_bm = 1 / rmsle_bm
w_rm = 1 / rmsle_rm
w_gb = 1 / rmsle_gb

soma_pesos = w_rm + w_gb

y_pred = (+ w_rm*y_pred_rm + w_gb*y_pred_gb) / soma_pesos

## Predição no Test Set (Ensemble Ponderado)

In [79]:
df_test = pd.read_csv(MODEL_DATA_DIR / "test_model.csv")
X_test = df_test[X_train.columns]

# y_pred_bm_test = model_best_bm.predict(X_test)
y_pred_rm_test = model_best_rm.predict(X_test)
y_pred_gb_test = model_best_gb.predict(X_test)

y_pred_test = (w_rm * y_pred_rm_test + w_gb * y_pred_gb_test
) / soma_pesos

submission = pd.DataFrame({
    "Id": df_test["ID"],
    "SalePrice": expm1(y_pred_test)
})

submission.to_csv(DATA_DIR / "submission.csv", index=False)
submission.head()

,Id,SalePrice
0,1461,126022.600007
1,1462,156029.930552
2,1463,193548.594186
3,1464,195030.071940
4,1465,204366.377219
